In [ ]:
# source: https://www.kaggle.com/discussions/general/74235
from google.colab import userdata
import os

os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

! python3 -m spacy download en_core_web_sm

# MBTI Datasets
# ! kaggle datasets download -d "zeyadkhalid/mbti-personality-types-500-dataset"
! kaggle datasets download -d "mazlumi/mbti-personality-type-twitter-dataset"
! kaggle datasets download -d "datasnaek/mbti-type" # included in first one

! unzip "*.zip"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Dataset URL: https://www.kaggle.com/datasets/mazlumi/mbti-personality-type-twitter-dataset
License(s): other
mbti-personality-type-twitter-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset URL: https://www.kaggle.com/datasets/datasnaek/mbti-type
License(s): CC0-1.0
mbti-type.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  mbti-type.zip
replace mbti_1.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n

Archive:  mbti-personality-types-500-dataset.zip
replace MBTI 500.csv? [y]es, [n]o

In [ ]:
import pandas as pd
import numpy as np

# Matplotlib, plt and cm modules
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Seaborn for plotting
import seaborn as sns

# Pritty print
from pprint import pprint

# for reading .env vars
import os

# For cleaning posts
import re

# For lemmanization
import spacy

DATASETS_PATH = './'

- No punctuations, stopwords, URLs
- Lemmatization (Running->Run, improves NLP)
- Reconstruct samples to be equal-sized chunks (500 words per sample)

In [ ]:
data = pd.read_csv(f'{DATASETS_PATH}mbti_1.csv')
data

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...
...,...,...
8670,ISFP,'https://www.youtube.com/watch?v=t8edHB_h908||...
8671,ENFP,'So...if this thread already exists someplace ...
8672,INTP,'So many questions when i do these things. I ...
8673,INFP,'I am very conflicted right now when it comes ...


In [ ]:
# source: https://www.kaggle.com/code/hadia150/advancedmbti-textclassification
def clean_post(text):
    """
    Clean social media text by removing noise
    """
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove post separator
    text = text.replace('|||', ' ')

    # Remove special characters but keep spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
data['clean_posts'] = data['posts'].apply(clean_post)

print(data.isnull().sum())

data

type           0
posts          0
clean_posts    0
dtype: int64


,type,posts,clean_posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,and intj moments sportscenter not top ten play...
1,ENTP,'I'm finding the lack of me in these posts ver...,im finding the lack of me in these posts very ...
2,INTP,'Good one _____ https://www.youtube.com/wat...,good one course to which i say i know thats my...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...",dear intp i enjoyed our conversation the other...
4,ENTJ,'You're fired.|||That's another silly misconce...,youre fired thats another silly misconception ...
...,...,...,...
8670,ISFP,'https://www.youtube.com/watch?v=t8edHB_h908||...,just because i always think of cats as fi doms...
8671,ENFP,'So...if this thread already exists someplace ...,soif this thread already exists someplace else...
8672,INTP,'So many questions when i do these things. I ...,so many questions when i do these things i wou...
8673,INFP,'I am very conflicted right now when it comes ...,i am very conflicted right now when it comes t...


In [ ]:
# source: https://www.kaggle.com/code/rajshreev/mbti-personality-predictor-using-machine-learning
def get_types(row):
    t=row['type']

#    I = 0; N = 0
#    T = 0; J = 0

    if t[0] == 'I': I = 'I'
    elif t[0] == 'E': I = 'E'
    else: print('I-E not found')

    if t[1] == 'N': N = 'N'
    elif t[1] == 'S': N = 'S'
    else: print('N-S not found')

    if t[2] == 'T': T = 'T'
    elif t[2] == 'F': T = 'F'
    else: print('T-F not found')

    if t[3] == 'J': J = 'J'
    elif t[3] == 'P': J = 'P'
    else: print('J-P not found')
    return pd.Series( {'IE':I, 'NS':N , 'TF': T, 'JP': J })

data = data.join(data.apply (lambda row: get_types (row),axis=1))
data.head(5)



,type,posts,clean_posts,IE,NS,TF,JP
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,and intj moments sportscenter not top ten play...,I,N,F,J
1,ENTP,'I'm finding the lack of me in these posts ver...,im finding the lack of me in these posts very ...,E,N,T,P
2,INTP,'Good one _____ https://www.youtube.com/wat...,good one course to which i say i know thats my...,I,N,T,P
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...",dear intp i enjoyed our conversation the other...,I,N,T,J
4,ENTJ,'You're fired.|||That's another silly misconce...,youre fired thats another silly misconception ...,E,N,T,J


In [43]:
# source: https://www.geeksforgeeks.org/machine-learning/python-pos-tagging-and-lemmatization-using-spacy/
nlp = spacy.load('en_core_web_sm')  # python -m spacy download en_core_web_sm

def lemmatize_text(text):
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc])

# ONLY RUN WHEN READY TO SAVE THE FILE
# data['lemmatize_clean_posts'] = data['clean_posts'].apply(lemmatize_text)

# THIS IS ONLY FOR THE EXAMPLE
data['clean_posts'].head(3).apply(lemmatize_text)

# Task: when cleaning and lemmatizing, download data to csv because it takes a while to run.

,clean_posts
0,and intj moment sportscenter not top ten play ...
1,I m find the lack of I in these post very alar...
2,good one course to which I say I know that s m...
